<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="560"/>

# Notebook 01 — Training a generative model for inverse design

*A guided tour, not an exercise sheet. Just run the cells top to bottom and read the prose between them.*

> **Colab users:** click **File ➜ Save a copy in Drive** before editing so your changes persist.

## Where we are in the workshop

In **Notebook 00** we pinned down what a `beams2d` design actually *is*: a 50×100 grid of material densities, scored by a physics simulator under a scenario `(volfrac, rmin, forcedist, overhang_constraint)`, with a classical topology optimiser as the baseline to beat.

That baseline works — and for `beams2d` it finishes in seconds. But it has two properties that get painful as soon as you leave the toy regime:

1. **Every new scenario starts from scratch.** The optimiser has no memory. If tomorrow a colleague hands you a thousand new `(volfrac, forcedist)` pairs, you pay the full iterative cost a thousand times.
2. **Every scenario returns one answer.** The optimiser converges to a single design. If you want to *explore* the design space — "give me five plausible beams for this load" — the optimiser can't help.

A **generative model** changes both of those. Train it once on a dataset of `(scenario, design)` pairs, and at inference time you hand it a scenario plus a random seed and it hands back a design in milliseconds. Different seeds give different designs.

That's the pitch. The *question* is whether those fast designs are any good. This notebook builds the generator. **Notebook 02** is where we find out.

## What this notebook is — and isn't

This notebook is **not** a tutorial on GAN architectures. We're not going to stare at layer diagrams or debate activation functions. EngiOpt already ships a catalogue of conditional generators (CGANs, diffusion models, VAEs, …) — we'll just pick one off the shelf and use it.

The point we *do* want to make is this: **hooking a generative model onto an EngiBench problem takes almost no glue code.** The benchmark hands you a dataset in the exact format supervised training expects. The model hands you designs in the exact format the simulator expects. There's no adapter layer in between.

By the end of this notebook you'll have:

- a trained conditional generator for `beams2d`,
- a folder of artifacts (generated designs, matched baselines, scenarios used) that Notebook 02 picks up and evaluates,
- a clearer feel for *why* we're about to spend a whole notebook on metrics.

## Install dependencies (Colab / fresh env only)

Skip this if your local environment already has `engibench` and `engiopt` installed.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # flip to True to force install locally

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[all]", "matplotlib", "tqdm"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    try:
        import torch  # noqa: F401
    except Exception:
        _pip(["torch", "torchvision"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch as th

from engibench.utils.all_problems import BUILTIN_PROBLEMS
from engiopt.cgan_cnn_2d.cgan_cnn_2d import Generator as CGAN2DGenerator
from engiopt.workshops.dcc26.notebook_helpers import (
    TrainingConfig,
    WorkshopGenerator,
    generate_designs,
    show_gen_vs_baseline,
    show_training_curve,
    show_training_progression,
    train_supervised_generator,
)

SEED = 7
random.seed(SEED); np.random.seed(SEED); th.manual_seed(SEED)

if th.cuda.is_available():
    DEVICE = th.device("cuda")
elif th.backends.mps.is_available():
    DEVICE = th.device("mps")
else:
    DEVICE = th.device("cpu")
print("Device:", DEVICE)

---
## 1 — *Load the problem — exactly the way Notebook 00 did*

Nothing new here. We ask for `beams2d` by name, and the problem object hands us everything we need: the design shape, the names of the conditions, and a train/val/test dataset of `(scenario, design)` pairs already pre-split for us.

Notice what we *didn't* have to do: write a data loader, define a schema, decide on a train/test split, normalise conditions, or figure out what a design even looks like. The benchmark already answered all of those.

In [ ]:
problem = BUILTIN_PROBLEMS["beams2d"](seed=SEED)
train_ds = problem.dataset["train"]
test_ds = problem.dataset["test"]

condition_keys = problem.conditions_keys
design_shape = problem.design_space.shape
n_conds = len(condition_keys)

print(f"Design shape   : {design_shape}")
print(f"Condition keys : {condition_keys}")
print(f"Train examples : {len(train_ds):,}")
print(f"Test examples  : {len(test_ds):,}")

---
## 2 — *Reshape the dataset into tensors the model can eat*

The benchmark dataset already pairs each design with its scenario. All we need to do is stack the columns into NumPy arrays so PyTorch can batch them. Two lines of real work.

The only subtlety is that we rescale designs from `[0, 1]` (the physics convention — 0 = void, 1 = solid) to `[-1, 1]` (the neural-network convention — matches the `tanh` output of the generator we'll use below). No physics changes; we're just matching conventions on the boundary.

In [ ]:
# (N, n_conds) float array — one row per training sample, one column per scenario field.
conds_np = np.stack(
    [np.array(train_ds[k]).astype(np.float32) for k in condition_keys],
    axis=1,
)

# (N, H, W) float array, rescaled from [0, 1] to [-1, 1].
designs_np = np.array(train_ds["optimal_design"]).astype(np.float32)
targets_np = designs_np * 2.0 - 1.0

print(f"conditions: {conds_np.shape},  range [{conds_np.min():.2f}, {conds_np.max():.2f}]")
print(f"targets   : {targets_np.shape}, range [{targets_np.min():.2f}, {targets_np.max():.2f}]")

That's the whole data-prep step. Compare that to the usual ML-tutorial pain of "find a dataset, clean it, split it, align it with labels." Pinning the benchmark contract in Notebook 00 is exactly what collapsed that work into three lines here.

---
## 3 — *Pick a generator off the EngiOpt shelf*

EngiOpt ships a catalogue of conditional generators for 2D problems. We'll use the CGAN-CNN generator (`engiopt.cgan_cnn_2d.Generator`) as a stand-in for "any off-the-shelf conditional image model." The exact architecture is *not* the lesson of this notebook — treat it as a black box with one job: map `(noise, conditions) → design`.

```
noise ∈ ℝ^32 ───┐
                 ├── [ conditional CNN generator ] ──→ design ∈ [0, 1]^{50×100}
conditions ∈ ℝ^4 ─┘
```

We wrap it in `WorkshopGenerator` so callers can pass plain 2D tensors instead of the 4D tensors the underlying CNN expects. That wrapper is two lines of reshaping.

In [ ]:
LATENT_DIM = 32  # size of the random noise vector per sample

cnn_gen = CGAN2DGenerator(
    latent_dim=LATENT_DIM,
    n_conds=n_conds,
    design_shape=design_shape,
)
model = WorkshopGenerator(cnn_gen).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Generator ready: {n_params:,} parameters, input = noise({LATENT_DIM}) + conditions({n_conds})")

One import, one constructor call. That's the integration point between EngiOpt and EngiBench.

---
## 4 — *Train it, supervised, against the optimiser's answers*

The training loop is the simplest thing that could work: for each batch, feed random noise plus real scenarios to the generator and ask it to match the optimiser's design on that scenario (MSE loss). No adversarial training, no diffusion schedule — just supervised regression on a benchmark dataset.

This is deliberately a *weak* generative model. Real methods do better. We're using the simplest possible recipe because the point of the notebook is the *plumbing*, not the performance.

> **Heads-up:** ~1–2 minutes on a GPU, ~5–10 minutes on CPU. Bump `EPOCHS` later if you want sharper designs.

In [ ]:
EPOCHS = 10

rng = np.random.default_rng(SEED)
snap_idx = rng.choice(len(test_ds), size=4, replace=False)
snap_conds = np.stack(
    [np.array(test_ds[k])[snap_idx].astype(np.float32) for k in condition_keys],
    axis=1,
)
snap_baselines = np.array(test_ds["optimal_design"])[snap_idx].astype(np.float32)

train_cfg = TrainingConfig(
    latent_dim=LATENT_DIM,
    epochs=EPOCHS,
    batch_size=64,
    lr=2e-4,
    device=DEVICE,
    snapshot_at_epochs=[1, max(1, EPOCHS // 2), EPOCHS],
)

result = train_supervised_generator(
    model, conds_np, targets_np,
    config=train_cfg,
    snapshot_conditions=snap_conds,
)
losses = result["losses"]
snapshots = result["snapshots"]

print(f"\nFinal loss after {EPOCHS} epochs: {losses[-1]:.5f}")

### Sanity-check the loss curve

A training loss that goes down is *necessary* but very much not *sufficient*. A low MSE means the generator is pixel-matching the dataset; it says nothing about whether the designs are physically valid, let alone stiff. We'll come back to that.

In [ ]:
show_training_curve(losses)

### Watch the designs emerge

At epoch 1 the generator outputs essentially noise. Halfway through training you start seeing dark blobs where material should go. By the final epoch the shape roughly tracks the ground-truth beam in the bottom row — same conditions, drawn from the test set, never shown during training.

In [ ]:
show_training_progression(snapshots, baseline_designs=snap_baselines, n_show=4)

---
## 5 — *Use it to generate designs for unseen scenarios*

The whole premise was "train once, generate instantly." Time to cash that in. We grab a batch of scenarios from the held-out test set — scenarios the model has never been trained on — and ask the generator for a design on each one.

No optimisation loop. No FEM. Just a forward pass.

In [ ]:
N_SAMPLES = 24

test_idx = rng.choice(len(test_ds), size=N_SAMPLES, replace=False)
test_conds_np = np.stack(
    [np.array(test_ds[k])[test_idx].astype(np.float32) for k in condition_keys],
    axis=1,
)
baseline_designs = np.array(test_ds["optimal_design"])[test_idx].astype(np.float32)

gen_designs = generate_designs(
    model, test_conds_np, latent_dim=LATENT_DIM, device=DEVICE,
)

print(f"Generated {gen_designs.shape[0]} designs of shape {gen_designs.shape[1:]} "
      f"in a single forward pass.")

### Generated vs. baseline — side by side

Top row: the generator's output. Bottom row: the optimiser's output for the same scenario. Both are attempts to answer the same question.

Stare at these and a few things jump out:

- **Blurriness.** The generator hedges. MSE loss rewards the average of all plausible designs for a given scenario, and the average of two valid truss topologies is an invalid blur. That's a loss-function problem, not a benchmark problem.
- **Condition sensitivity.** Do the generated designs actually *change* when the scenario changes, or does the model output something close to "the dataset mean"? Eyeballing this is hard; measuring it is what Notebook 02 is for.
- **Plausibility.** Some of these look like beams. Some don't. A picture can't tell you whether the design is stiff under load, or whether the material budget is respected. *You need a simulator for that.*

In [ ]:
conditions_records = [
    {k: float(test_conds_np[i, j]) for j, k in enumerate(condition_keys)}
    for i in range(N_SAMPLES)
]

show_gen_vs_baseline(gen_designs, baseline_designs, conditions_records)

---
## 6 — *Export artifacts for Notebook 02*

Notebook 02 will load these three files and run the physics simulator on every generated design. We save them now so the evaluation notebook can start cold — no notebook-to-notebook Python state required.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
ARTIFACT_DIR = (
    Path("/content/dcc26_artifacts") if IN_COLAB
    else Path("workshops/dcc26/artifacts")
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.save(ARTIFACT_DIR / "generated_designs.npy", gen_designs)
np.save(ARTIFACT_DIR / "baseline_designs.npy", baseline_designs)
with open(ARTIFACT_DIR / "conditions.json", "w") as f:
    json.dump(conditions_records, f, indent=2)

print(f"Saved artifacts to {ARTIFACT_DIR}:")
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {p.name}")

---
## Putting it together

Every box in the "generative model for inverse design" picture was filled by one of two sides — EngiBench or EngiOpt:

| What you need                              | Who provides it | One-line access |
|--------------------------------------------|-----------------|-----------------|
| The problem definition                     | EngiBench       | `BUILTIN_PROBLEMS["beams2d"]()` |
| `(scenario, design)` training pairs        | EngiBench       | `problem.dataset["train"]` |
| The shape and range of a valid design      | EngiBench       | `problem.design_space` |
| The scenario schema                        | EngiBench       | `problem.conditions_keys` |
| A conditional generator architecture       | EngiOpt         | `engiopt.cgan_cnn_2d.Generator` |
| A training recipe                          | EngiOpt         | `train_supervised_generator(...)` |
| The yardstick to score generated designs   | EngiBench       | `problem.simulate(...)` (Notebook 02) |
| The validity test                          | EngiBench       | `problem.check_constraints(...)` (Notebook 02) |

The only code we wrote in this notebook was **plumbing** — stacking columns into arrays, reshaping tensors, saving `.npy` files. That's the story: once a benchmark pins down the contract, integrating an ML method against it is mostly a matter of connecting pipes that already exist.

---
## Reflect before moving on

1. Look at the side-by-side figure. If you had to write a short paragraph in a paper claiming your generator "works," what evidence in that figure would you cite — and what would a sceptical reviewer push back on?
2. We trained the model to minimise pixel-MSE against the optimiser's designs. Name one thing a *lower* MSE could improve, and one thing a lower MSE says *nothing* about.
3. The generator runs in milliseconds; the optimiser takes seconds. At what break-even number of scenarios does "train a generator once" start paying off compared to just running the optimiser each time?

## Next

In **Notebook 02** we run the physics simulator on every design this notebook produced, check them against the benchmark's constraints, and turn the intuitions above into actual numbers — objective gaps, feasibility rates, diversity proxies. *That* is what lets us say whether the generator "works."